In [2]:
import pandas as pd
import numpy as np
import datetime as dt

In [3]:
df = pd.read_excel(r'C:\Backup_D\Dolanan_VScode\irlo.xlsx', sheet_name=4)
df.head()

,Date,Order ID,Customer ID,Weight,Piece,Piece Type,Service Speed,Service,Sales,Region
0,2025-09-01,1001,INE729,0.96,NaN,NaN,Reguler,Complete Wash,24.00,GJM
1,2025-09-01,1002,PER783,3.38,NaN,NaN,Reguler,Complete Wash,27.04,GJM
2,2025-09-01,1002,PER783,0.00,Small Bed Sheet Set,Bed Sheet,Piece,Piece,15.00,GJM
3,2025-09-01,1003,PUR183,4.49,NaN,NaN,Reguler,Complete Wash,35.92,GJM
4,2025-09-01,1003,PUR183,0.00,Small Bed Sheet Set,Bed Sheet,Piece,Piece,15.00,GJM


In [4]:
df2 = df[['Date', 'Order ID', 'Customer ID', 'Sales']].copy()
df2.head()

,Date,Order ID,Customer ID,Sales
0,2025-09-01,1001,INE729,24.00
1,2025-09-01,1002,PER783,27.04
2,2025-09-01,1002,PER783,15.00
3,2025-09-01,1003,PUR183,35.92
4,2025-09-01,1003,PUR183,15.00


In [5]:
df_user = (
    df2
    .groupby('Customer ID', as_index=False)
    .agg(
        order_cnt=('Order ID', 'nunique'),
        max_order_date=('Date', 'max'),
        total_order_value=('Sales', 'sum')
    )
)

df_user.head()

,Customer ID,order_cnt,max_order_date,total_order_value
0,ADE488,4,2026-01-17,136.16
1,ADH374,30,2026-02-22,1107.28
2,ADH375,5,2026-02-04,216.24
3,ADI693,5,2026-02-16,143.36
4,ADJ966,5,2025-12-29,224.76


In [6]:
today = df2['Date'].max()
df_user['d_sinlas'] = (today - df_user['max_order_date']).dt.days

df_user.head()

,Customer ID,order_cnt,max_order_date,total_order_value,d_sinlas
0,ADE488,4,2026-01-17,136.16,42
1,ADH374,30,2026-02-22,1107.28,6
2,ADH375,5,2026-02-04,216.24,24
3,ADI693,5,2026-02-16,143.36,12
4,ADJ966,5,2025-12-29,224.76,61


Create RFM

In [7]:
df_user['recency_score'] = (
    pd.cut(
        df_user['d_sinlas'],
        bins=[
               df_user['d_sinlas'].min(),
               np.percentile(df_user['d_sinlas'], 20),
               np.percentile(df_user['d_sinlas'], 40),
               np.percentile(df_user['d_sinlas'], 60),
               np.percentile(df_user['d_sinlas'], 80),
               df_user['d_sinlas'].max()
        ],
        labels=[5,4,3,2,1],
        include_lowest=True
    )
    .astype(int)
)

df_user.head()

,Customer ID,order_cnt,max_order_date,total_order_value,d_sinlas,recency_score
0,ADE488,4,2026-01-17,136.16,42,1
1,ADH374,30,2026-02-22,1107.28,6,4
2,ADH375,5,2026-02-04,216.24,24,2
3,ADI693,5,2026-02-16,143.36,12,3
4,ADJ966,5,2025-12-29,224.76,61,1


In [8]:
df_user['frequency_score'] = (
    pd.cut(
        df_user['order_cnt'],
        bins=[
               0,
               np.percentile(df_user['order_cnt'], 20),
               np.percentile(df_user['order_cnt'], 40),
               np.percentile(df_user['order_cnt'], 60),
               np.percentile(df_user['order_cnt'], 80),
               df_user['order_cnt'].max()
        ],
        labels=[1,2,3,4,5],
        include_lowest=True
    )
    .astype(int)
)

df_user.head()

,Customer ID,order_cnt,max_order_date,total_order_value,d_sinlas,recency_score,frequency_score
0,ADE488,4,2026-01-17,136.16,42,1,1
1,ADH374,30,2026-02-22,1107.28,6,4,4
2,ADH375,5,2026-02-04,216.24,24,2,1
3,ADI693,5,2026-02-16,143.36,12,3,1
4,ADJ966,5,2025-12-29,224.76,61,1,1


In [9]:
df_user['monetary_score'] = (
    pd.cut(
        df_user['total_order_value'],
        bins=[
               df_user['total_order_value'].min(),
               np.percentile(df_user['total_order_value'], 20),
               np.percentile(df_user['total_order_value'], 40),
               np.percentile(df_user['total_order_value'], 60),
               np.percentile(df_user['total_order_value'], 80),
               df_user['total_order_value'].max()
        ],
        labels=[1,2,3,4,5],
        include_lowest=True
    )
    .astype(int)
)

df_user.head()

,Customer ID,order_cnt,max_order_date,total_order_value,d_sinlas,recency_score,frequency_score,monetary_score
0,ADE488,4,2026-01-17,136.16,42,1,1,1
1,ADH374,30,2026-02-22,1107.28,6,4,4,4
2,ADH375,5,2026-02-04,216.24,24,2,1,2
3,ADI693,5,2026-02-16,143.36,12,3,1,1
4,ADJ966,5,2025-12-29,224.76,61,1,1,2


In [10]:
df_user2 = df_user[['Customer ID', 'recency_score', 'frequency_score', 'monetary_score']].copy()
df_user2.head()

,Customer ID,recency_score,frequency_score,monetary_score
0,ADE488,1,1,1
1,ADH374,4,4,4
2,ADH375,2,1,2
3,ADI693,3,1,1
4,ADJ966,1,1,2


In [11]:
df_user2['RFM_score_segment'] = (
    df_user2['recency_score'].astype(str)
    +
    df_user2['frequency_score'].astype(str)
    +
    df_user2['monetary_score'].astype(str)
)

df_user2.head()

,Customer ID,recency_score,frequency_score,monetary_score,RFM_score_segment
0,ADE488,1,1,1,111
1,ADH374,4,4,4,444
2,ADH375,2,1,2,212
3,ADI693,3,1,1,311
4,ADJ966,1,1,2,112


In [12]:
segment_mapping = {
    r'[1-2][1-2][1-5]': 'Hibernating',
    r'[1-2][3-4][1-5]': 'At Risk',
    r'[1-2][5][1-5]': 'Can\'t Loose',
    r'[3][1-2][1-5]': 'About to Sleep',
    r'[3][3][1-5]': 'Need Attention',
    r'[3-4][4-5][1-5]': 'Loyal Customers',
    r'[4][1][1-5]': 'Promising',
    r'[5][1][1-5]': 'New Customers',
    r'[4-5][2-3][1-5]': 'Potential Loyalists',
    r'[5][4-5][1-5]': 'Champions'
}

df_user2['Segment'] = (
    df_user2['RFM_score_segment']
    .replace(segment_mapping, regex=True)
)

df_user2.head()

,Customer ID,recency_score,frequency_score,monetary_score,RFM_score_segment,Segment
0,ADE488,1,1,1,111,Hibernating
1,ADH374,4,4,4,444,Loyal Customers
2,ADH375,2,1,2,212,Hibernating
3,ADI693,3,1,1,311,About to Sleep
4,ADJ966,1,1,2,112,Hibernating


Create Visualization

In [13]:
import plotly.express as px

In [14]:
df_grouped = (
    df_user2
    .groupby('Segment')
    .size()
    .reset_index(name='customer_count')
)

df_grouped

,Segment,customer_count
0,About to Sleep,33
1,At Risk,68
2,Can't Loose,5
3,Champions,68
4,Hibernating,102
5,Loyal Customers,77
6,Need Attention,18
7,New Customers,12
8,Potential Loyalists,58
9,Promising,8


In [30]:
fig = px.treemap(
    df_grouped,
    path=['Segment'],
    values='customer_count',
    title='<b>RFM Customer Segments Distribution</b>',
    color='customer_count',
    color_continuous_scale='Tealgrn',
)

fig.update_traces(
    textinfo='label+value+percent root',
    texttemplate='<b>%{label}</b><br>Count: %{value:,}<br>Share: %{percentRoot:.1%}',
    textposition='middle center',
)

fig.update_layout(
    margin=dict(t=50, l=55, r=25, b=25),
    title_font=dict(size=18, family='Arial'),
)

fig.show()